# Generate GRPO Training Data — Quartermaster Environment

This notebook generates GRPO training prompts by running inference and saving replay data.
For each step in each episode, it saves the observation + prior action history needed to replay env state.

**Output:** `grpo_data.jsonl`

The GRPO trainer loads this as prompts. For each prompt, the model generates G completions online,
each is parsed into an action, the env is replayed to that step, stepped once, and the real reward is returned.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install "openenv-core[core]>=0.2.0" fastapi uvicorn pydantic numpy openai python-dotenv matplotlib

## 1. Configuration

In [ ]:
import os
import json
import time
import logging

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

# --- Configuration ---
API_BASE_URL = os.getenv("API_BASE_URL") or "https://router.huggingface.co/v1"
API_KEY = os.getenv("API_KEY") or os.getenv("HF_TOKEN") or os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME") or "Qwen/Qwen3-32B"
OUTPUT_FILE = "grpo_data.jsonl"
NUM_EPISODES = 1       # Episodes per task
TASKS_TO_RUN = ["easy", "medium", "hard"]

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("generate_grpo_data")

print(f"Model: {MODEL_NAME}")
print(f"API: {API_BASE_URL}")
print(f"Episodes per task: {NUM_EPISODES}")
print(f"Tasks: {TASKS_TO_RUN}")
print(f"Output: {OUTPUT_FILE}")

## 2. Load Environment & Inference Utilities

In [ ]:
from server.inventory_env import InventoryEnvironment
from models import InventoryAction
from inference import SYSTEM_PROMPT, format_observation, parse_action

client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)

# Sanity check
env = InventoryEnvironment("easy")
obs = env.reset()
print(f"Environment loaded. Day {obs.current_day}/{obs.total_days}, Cash: ${obs.total_cash:.0f}")

## 3. Action Serialization Helper

In [ ]:
def action_to_dict(action):
    """Convert InventoryAction to a minimal serializable dict."""
    d = {}
    if action.buy_quantities:
        d["buy_quantities"] = action.buy_quantities
    if action.delivery_methods:
        d["delivery_methods"] = action.delivery_methods
    if action.liquidate:
        d["liquidate"] = action.liquidate
    if action.price_multipliers:
        d["price_multipliers"] = action.price_multipliers
    if action.notes_to_self:
        d["notes_to_self"] = action.notes_to_self
    if action.weekly_plan is not None:
        d["weekly_plan"] = action.weekly_plan
    if action.take_loan:
        d["take_loan"] = True
    return d

## 4. Episode Runner with Replay Data

For each step, saves:
- `observation`: the formatted text (what the model sees)
- `task_name`: easy/medium/hard
- `prior_actions`: JSON list of action dicts for days 1..N-1 (to replay env state)
- `day`: current day number

In [ ]:
def run_episode(client, task_name, episode_num):
    """Run one episode, collect GRPO training data for every step."""
    env = InventoryEnvironment(task_name)
    obs = env.reset()

    examples = []
    prior_actions = []
    rewards = []
    ep_start = time.time()

    for day in range(1, env.max_days + 1):
        if obs.done:
            break

        obs_text = format_observation(obs)

        # Save this step's training example
        examples.append({
            "observation": obs_text,
            "task_name": task_name,
            "prior_actions": json.dumps(prior_actions),
            "day": day,
            "episode": episode_num,
        })

        # Get action from baseline model
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs_text},
        ]

        step_start = time.time()
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.6,
                max_completion_tokens=800,
                stream=False,
            )
            response_text = completion.choices[0].message.content or ""
        except Exception as exc:
            log.warning(f"[{task_name}] ep{episode_num} day{day}: API error: {exc}")
            response_text = "{}"

        action = parse_action(response_text)
        action_dict = action_to_dict(action)

        obs = env.step(action)
        rewards.append(obs.reward)
        prior_actions.append(action_dict)

        step_ms = (time.time() - step_start) * 1000
        log.info(
            f"[{task_name}] ep{episode_num} day{day:02d}: "
            f"reward={obs.reward:+.2f} profit=${obs.total_profit:.0f} "
            f"cash=${obs.total_cash:.0f} "
            f"violations={len(obs.directive_violations_last_step)} "
            f"({step_ms:.0f}ms)"
        )

    ep_time = time.time() - ep_start
    avg_reward = sum(rewards) / len(rewards) if rewards else 0
    log.info(
        f"[{task_name}] ep{episode_num} DONE: {len(examples)} steps, "
        f"avg_reward={avg_reward:.3f}, final_profit=${obs.total_profit:.0f}, "
        f"time={ep_time:.1f}s"
    )
    return examples, rewards

## 5. Generate Data — All Tasks

In [ ]:
all_examples = []
all_rewards = {}
run_start = time.time()

for task_name in TASKS_TO_RUN:
    task_rewards = []
    for ep in range(1, NUM_EPISODES + 1):
        log.info(f"--- [{task_name}] Episode {ep}/{NUM_EPISODES} ---")
        examples, rewards = run_episode(client, task_name, ep)
        all_examples.extend(examples)
        task_rewards.extend(rewards)

    all_rewards[task_name] = task_rewards
    if task_rewards:
        log.info(
            f"[{task_name}] SUMMARY: {len(task_rewards)} steps, "
            f"avg_reward={sum(task_rewards)/len(task_rewards):.3f}, "
            f"min={min(task_rewards):.3f}, max={max(task_rewards):.3f}"
        )

total_time = time.time() - run_start
print(f"\nTotal: {len(all_examples)} GRPO prompts in {total_time:.1f}s")

## 6. Save to JSONL

In [ ]:
with open(OUTPUT_FILE, "a") as f:
    for ex in all_examples:
        f.write(json.dumps(ex) + "\n")

print(f"Appended {len(all_examples)} prompts to {OUTPUT_FILE}")

## 7. Data Stats & Visualization

In [ ]:
# Save reward stats
stats_file = OUTPUT_FILE.replace(".jsonl", "_stats.json")
stats = {}
for task_name in TASKS_TO_RUN:
    task_examples = [e for e in all_examples if e["task_name"] == task_name]
    if task_examples and all_rewards.get(task_name):
        stats[task_name] = {
            "count": len(task_examples),
            "mean_reward": sum(all_rewards[task_name]) / len(all_rewards[task_name]),
            "min_reward": min(all_rewards[task_name]),
            "max_reward": max(all_rewards[task_name]),
        }
with open(stats_file, "w") as f:
    json.dump(stats, f, indent=2)

print("=== GRPO DATA STATS ===")
for task_name in TASKS_TO_RUN:
    task_examples = [e for e in all_examples if e["task_name"] == task_name]
    if task_examples:
        days = [e["day"] for e in task_examples]
        print(f"  {task_name}: {len(task_examples)} steps, days {min(days)}-{max(days)}")
print(f"Stats saved to {stats_file}")

In [ ]:
import matplotlib.pyplot as plt

task_colors = {"easy": "#22c55e", "medium": "#f59e0b", "hard": "#ef4444"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward by day per task
ax = axes[0]
for task_name in TASKS_TO_RUN:
    if task_name in all_rewards and all_rewards[task_name]:
        ax.plot(range(1, len(all_rewards[task_name]) + 1), all_rewards[task_name],
                color=task_colors.get(task_name, "#888"), alpha=0.7, linewidth=1.5, label=task_name)
ax.set_xlabel("Day")
ax.set_ylabel("Reward")
ax.set_title("Baseline Reward by Day")
ax.legend()
ax.grid(True, alpha=0.3)

# Reward histogram
ax = axes[1]
for task_name in TASKS_TO_RUN:
    if task_name in all_rewards and all_rewards[task_name]:
        ax.hist(all_rewards[task_name], bins=30, alpha=0.5,
                color=task_colors.get(task_name, "#888"), label=task_name, edgecolor="#000")
ax.set_xlabel("Reward")
ax.set_ylabel("Count")
ax.set_title("Baseline Reward Distribution")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("GRPO Data — Baseline Performance", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()